# Generowanie opisów obrazów – Show and Tell (2014)

Projekt realizowany w ramach przedmiotu _Analiza danych obrazowych i multimedialnych_.  
Zespół projektowy: Michał Suchocki, Piotr Szczerba, Krzysztof Rudnik, Krzysztof Dąbrowski  
Realizacja: 2026-06-02

<a href="https://github.com/christopher-dabrowski/Show-and-Tell" target="_blank" rel="noopener noreferrer">
  <div style="display:inline-flex; align-items:center; gap:8px;">
    <img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png" alt="GitHub" width="16" style="vertical-align:middle;" />
    Repozytorium
    <img alt="GitHub commits since tagged version" src="https://img.shields.io/github/commits-since/christopher-dabrowski/Show-and-Tell/forked?style=for-the-badge">
  </div>
</a>


# Spis treści
<!-- TODO: Uzupełnić spis treści na koniec -->

- [Omówienie metody Show and Tell](#omowienie-metody-show-and-tell)
- [Omówienie wykorzystanej implementacji](#omowienie-wykorzystanej-implementacji)
- [Wyniki działania](#wyniki-dzialania)
- [Krytyczna analiza](#krytyczna-analiza)
- [Wnioski](#wnioski)

# Omówienie metody Show and Tell
<!-- krótko, zwięźle: do czego słuzy, na czym polega, podać linki do bardziej szczegółowych omówień, tutoriali, do repozytoriów kodu -->

Model Show and Tell jest jednym z pierwszych skutecznych podejść do automatycznego generowania opisów tekstowych na podstawie obrazów. Architektura **łączy sieć konwolucyjną**, która pełni rolę ekstraktora cech wizualnych, **z siecią rekurencyjną** odpowiedzialną za generowanie sekwencji słów opisujących obraz.
Wektor cech obrazu jest wykorzystywany jako wejście modelu językowego, który przewiduje kolejne słowa opisu w sposób sekwencyjny. Dzięki temu możliwe jest automatyczne tworzenie zdań opisujących zawartość sceny, obecne obiekty oraz relacje między nimi. Model został wytrenowany na dużych zbiorach zawierających obrazy wraz z opisami tekstowymi, co pozwoliło mu nauczyć się powiązań między reprezentacją wizualną a językiem naturalnym. Podejście to zapoczątkowało intensywny rozwój metod multimodalnych łączących analizę obrazu i przetwarzanie języka naturalnego. Zadanie generowania opisów obrazów jest szczególnie wymagające, ponieważ wymaga zarówno rozpoznania obiektów, jak i zrozumienia ich kontekstu. Analiza tego modelu pozwala zrozumieć sposób integracji reprezentacji wizualnych z modelami sekwencyjnymi oraz podstawy współczesnych systemów multimodalnych.

Szczegółowy opis metody przedstawia oryginalna publikacja [Show and Tell: A Neural Image Caption Generator](https://arxiv.org/pdf/1411.4555).
Nasze eksperymenty prowadziliśmy w oparciu o tutorial [a PyTorch Tutorial to Image Captioning](https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning). Zawiera ciekawe i rzetelne wyjaśnienie metody Show and Tell rozbudowanej o mechanizm uwagi.


![Schemat działania Show and Tell](img/overview_fig_2.png)  
Schemat działania metody Show and Tell  
_Źródło: [Show and Tell: A Neural Image Caption Generator](https://arxiv.org/pdf/1411.4555)_

## Ekstraktor cech

Ekstraktor cech, nazywany również koderem, odpowiada za przetworzenie obrazu wejściowego na tensor zawierający informacje potrzebne to wygenerowania opisu przez dalszy model. Zazwyczaj używa się do tego sieci konwolucyjnej, ponieważ stan technologii do tego czasu wykazał, że sieci tego typu są skuteczne w klasyfikacji obrazów, autorzy założyli, że w takim razie sieci konwolucyjne będą również skuteczne w ekstrakcji cech potrzebnych do generowania opisów.

Przykładowo dla sieci konwolucyjnej ResNet-101 transformacja wygląda następująco:

![Ekstraktor cech](img/encoder_pl.png)

W przeciwieństwie do typowego zastosowania sieci konwolucyjnej, gdzie na końcu znajduje się warstwa klasyfikacyjna, w modelu Show and Tell wykorzystuje się wyjście z warstwy konwolucyjnej przed warstwą klasyfikacyjną. Wystarczy **usunąć warstwę klasyfikacyjną**.

## Dekoder

Rolą dekodera jest wygenerowanie opisu tekstowego na podstawie cech obrazu. W modelu Show and Tell wykorzystuje się do tego sieć rekurencyjną, a dokładniej LSTM (Long Short-Term Memory).

![Decoder bez uwagi](./img/decoder_no_att_pl.png)

Bazowa praca nie uwzględnia mechanizmu uwagi, jednak tutorial, z którego korzystaliśmy korzysta z tego podejścia, więc w naszych eksperymentach wykorzystujemy dekoder z mechanizmem uwagi.

## Beam Search

Zamiast wybierania jedynie najbardziej prawdopodobnego słowa na każdym kroku, użyta została metoda Beam Search, która pozwala na eksplorację wielu potencjalnych sekwencji jednocześnie. W każdym kroku generowania opisu utrzymuje się k najlepszych częściowych tłumaczeń, zwanych hipotezami. Na końcu wybierana jest sekwencja o najwyższym łącznym prawdopodobieństwie, co prowadzi do znalezienia bardziej optymalnego opisu niż w podejściu zachłannym.

![Beam Search](./img/beam_search.png)

# Omówienie wykorzystanej implementacji

Implementacja wykorzystana do naszych eksperymentów bazowała na tutorialu [a PyTorch Tutorial to Image Captioning](https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning). Nasze repozytorium jest forkiem tego tutorialu.

Przygotowany model jest rozbudowany o mechanizm uwagi, który nie występuje w bazowej pracy [Show and Tell: A Neural Image Caption Generator](https://arxiv.org/pdf/1411.4555). Na tak rozbudowanym modelu przeprowadziliśmy nasze badania. Mechanizm uwagi wykracza poza bazową metodę, jest jednak dobrze opisany w sekcji [Attention tutorialu](https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning#attention), z którego korzystaliśmy.

## Uruchomienie projektu

Do zarządzania wersją python oraz pakietami użyte zostało narzędzie [uv](https://docs.astral.sh/uv/). Jest napisane w Rust i jest znacznie szybsze od pozostałych opcji 😎

1. Zainstaluj `uv` zgodnie z [instrukcjami na stronie projektu](https://docs.astral.sh/uv/getting-started/installation/) (jeśli nie zainstalowałeś go wcześnie za pomocą `mise`).
2. Uruchom `uv sync` w katalogu projektu, aby zainstalować zależności i utworzyć środowisko wirtualne.
3. Jeżeli masz kartę graficzną serii RTX 50xx, zainstaluj odpowiednią wersję PyTorch: `uv pip install --force-reinstall --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128`
4. Pobierz [wagi modelu](https://drive.google.com/open?id=189VY65I_n4RTpQnmLGj7IzVnOF6dmePC) do katalogu `checkpoints/`, jeśli nie chcesz samodzielnie trenować modelu.
5. Pobierz i rozpakuj przygotowane dane [treningowe](http://images.cocodataset.org/zips/train2014.zip) i [testowe](http://images.cocodataset.org/zips/val2014.zip) do katalogu `dataset/`.
6. Uruchom `uv run create_input_files.py` żeby wygenerować dane wejściowe dla modelu na podstawie pobranych danych.
7. Uruchom wybrany Jupyter Notebook korzystając z wirtualnego środowiska python jako kernela.

Bardziej rozbudowane instrukcje do pracy rozwojowej nad projektem opisaliśmy w [Readme](https://github.com/christopher-dabrowski/Show-and-Tell/blob/master/Readme.md).

## Struktura repozytorium

- Bazowe pliki tworzące model Show and Tell, takie jak `model.py`, `dataset.py`, `train.py` i `eval.py` pozwalające na trenowanie i testowanie modelu, są w bazowym katalogu repozytorium.
- `checkpoints/` zawiera zapisane wagi modelu po treningu. Z uwagi na duży rozmiar pliki te nie są przechowywane w repozytorium.
- `dataset/` zawiera przygotowane dane treningowe i testowe. Ze względu na duży rozmiar, podobnie jak w przypadku checkpointów, nie są one przechowywane w repozytorium.
- `ekperymenty/` zawiera notebooki z eksperymentami, w których testowaliśmy różne konfiguracje modelu i analizowaliśmy wyniki.
- `img/` zawiera obrazy używane w dokumentacji.
- `.github/workflows/` zawiera konfigurację GitHub Actions do automatycznej walidacji jakości kodu. Kod z wynikiem `pylint` poniżej 7 nie jest akceptowany.
- `pyproject.toml`, `uv.lock` i `.python-version` zawierają informacje o zależnościach projektu i użytej wersji Python, zarządzane przez `uv`.
- `.mise.toml` zawiera konfigurację narzędzia `mise`, które jest używane przygotowania narzędzi deweloperskich, takich jak `uv`.
- `.lefthook.yaml` zawiera konfigurację narzędzia `lefthook`, które jest używane do walidacji kodu przed każdym commitem.

## Implementacja modelu

Implementacja poszczególnych modeli i całego potoku przetwarzania danych jest szczegółowo opisana w sekcji [Implementation tutorialu a PyTorch Tutorial to Image Captioning](https://github.com/sgrvinod/a-PyTorch-Tutorial-to-Image-Captioning#implementation).

# Wyniki działania
<!-- TODO: (własnych - innych niż te z repozytorium kodu) - najpierw tych bezproblemowych -->

# Krytyczna analiza
<!-- TODO: (podsumowanie weryfikacji eksperymentalnej) -->

# Wnioski
<!-- głównie w zakresie stosowalności metody na podstawie własnych eksperymentów - kiedy się sprawdza, kiedy nie itd.  -->